# Exercise 11 — MovieLens Case Study

This notebook creates a compact case-study analysis of the MovieLens small dataset. It builds the bipartite graph, enriches it with node metadata, computes final metrics, exports artifacts, and summarizes the network's structural story.

## Goal
Use the Lecture 11 workflow to produce a polished analysis package for the MovieLens topic.

## Setup
Import packages, load MovieLens data, and prepare the graph construction pipeline.

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

output_dir = Path('exercises/muvodic/exercise_11')
output_dir.mkdir(parents=True, exist_ok=True)

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## Data Loading and Preprocessing
We load the MovieLens ratings and movie metadata, then prepare user/movie node labels and movie genres for graph enrichment.

In [2]:
data_dir = Path('../data/movielense/ml-latest-small')
ratings = pd.read_csv(data_dir / 'ratings.csv')
movies = pd.read_csv(data_dir / 'movies.csv')

ratings['user_node'] = ratings['userId'].astype(str).radd('user_')
ratings['movie_node'] = ratings['movieId'].astype(str).radd('movie_')
movies['movie_node'] = movies['movieId'].astype(str).radd('movie_')
movies['genres_list'] = movies['genres'].fillna('None').str.split('|')

print('Ratings rows:', len(ratings))
print('Movies rows:', len(movies))

Ratings rows: 100836
Movies rows: 9742


## Graph Construction
Build the bipartite user-movie graph and attach metadata for movie nodes.

In [3]:
G = nx.Graph()
G.add_nodes_from(ratings['user_node'].unique(), bipartite=0, node_type='user')
G.add_nodes_from(ratings['movie_node'].unique(), bipartite=1, node_type='movie')
G.add_edges_from(zip(ratings['user_node'], ratings['movie_node']), edge_type='rating')

movie_metadata = movies.set_index('movie_node')[['title', 'genres', 'genres_list']].to_dict('index')
for movie_node, attrs in movie_metadata.items():
    if movie_node in G:
        nx.set_node_attributes(G, {movie_node: attrs})

print(f'Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.')

Graph built with 10334 nodes and 100836 edges.


## Enrichment and Attributes
Add node attributes from metadata and compute movie genre counts for later interpretation.

In [4]:
for node in G.nodes():
    if G.nodes[node]['node_type'] == 'user':
        G.nodes[node]['label'] = node
    else:
        G.nodes[node]['label'] = G.nodes[node].get('title', node)

movie_genre_counts = {}
for _, row in movies.iterrows():
    for genre in row['genres_list']:
        movie_genre_counts[genre] = movie_genre_counts.get(genre, 0) + 1

movie_genre_counts = dict(sorted(movie_genre_counts.items(), key=lambda x: x[1], reverse=True))
print('Top genres:', list(movie_genre_counts.items())[:5])

Top genres: [('Drama', 4361), ('Comedy', 3756), ('Thriller', 1894), ('Action', 1828), ('Romance', 1596)]


## Final Metrics
Compute a compact set of metrics including density, connectivity, clustering, and centrality.

In [5]:
metrics = {}
metrics['node_count'] = G.number_of_nodes()
metrics['edge_count'] = G.number_of_edges()
metrics['density'] = nx.density(G)
metrics['connected_components'] = nx.number_connected_components(G)
metrics['average_clustering'] = nx.average_clustering(G)
metrics['user_count'] = sum(1 for _, d in G.nodes(data=True) if d['node_type'] == 'user')
metrics['movie_count'] = sum(1 for _, d in G.nodes(data=True) if d['node_type'] == 'movie')

largest_cc = max(nx.connected_components(G), key=len)
metrics['largest_component_size'] = len(largest_cc)
metrics['largest_component_fraction'] = len(largest_cc) / G.number_of_nodes()

degree_dict = dict(G.degree())
nx.set_node_attributes(G, degree_dict, 'degree')

pagerank = nx.pagerank(G, alpha=0.85)
nx.set_node_attributes(G, pagerank, 'pagerank')

metrics['top_degree'] = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)[:5]
metrics

ModuleNotFoundError: No module named 'scipy'

## Ranked Outputs
Create a final node ranking table for movies and users, then export the top entries.

In [ ]:
node_data = []
for node, attrs in G.nodes(data=True):
    node_data.append({
        'node': node,
        'node_type': attrs['node_type'],
        'label': attrs['label'],
        'degree': attrs['degree'],
        'pagerank': attrs['pagerank'],
        'genres': attrs.get('genres', '')
    })
node_df = pd.DataFrame(node_data)
movie_ranking = node_df[node_df['node_type'] == 'movie'].sort_values(['degree', 'pagerank'], ascending=[False, False]).head(20)
user_ranking = node_df[node_df['node_type'] == 'user'].sort_values(['degree', 'pagerank'], ascending=[False, False]).head(20)

movie_ranking.to_csv('exercises/muvodic/exercise_11/top_movie_ranking.csv', index=False)
user_ranking.to_csv('exercises/muvodic/exercise_11/top_user_ranking.csv', index=False)
node_df.to_csv('exercises/muvodic/exercise_11/full_node_ranking.csv', index=False)

movie_ranking.head(10)

## Final Visualization
Draw a compact presentation-style figure for the top movie hubs and their neighbors.

In [ ]:
top_movie_nodes = movie_ranking['node'].tolist()[:5]
hub_ego = nx.ego_graph(G, top_movie_nodes[0], radius=1)
pos = nx.spring_layout(hub_ego, seed=42, k=0.15)

plt.figure(figsize=(10, 10))
node_colors = []
node_sizes = []
for node in hub_ego.nodes():
    if node == top_movie_nodes[0]:
        node_colors.append('red')
        node_sizes.append(400)
    elif hub_ego.nodes[node]['node_type'] == 'movie':
        node_colors.append('orange')
        node_sizes.append(200)
    else:
        node_colors.append('skyblue')
        node_sizes.append(100)

nx.draw(hub_ego, pos, with_labels=False, node_color=node_colors, node_size=node_sizes, alpha=0.85)
labels = {node: hub_ego.nodes[node]['label'] for node in hub_ego.nodes() if hub_ego.nodes[node]['node_type'] == 'movie'}
nx.draw_networkx_labels(hub_ego, pos, labels=labels, font_size=8)
plt.title('MovieLens Hub Ego Network: Top Movie and Its Users')
plt.axis('off')
plt.savefig('exercises/muvodic/exercise_11/top_movie_hub_ego.png', dpi=200, bbox_inches='tight')
plt.show()

## Artifact Export
Save a compact metrics summary and a GraphML file for the enriched graph.

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
with open(output_dir / 'metrics_summary.txt', 'w') as f:
    for key, value in metrics.items():
        f.write(f"{key}: {value}\n")

nx.write_gml(G, output_dir / 'movielens_case_study.gml')
print('Artifacts exported: metrics_summary.txt, top_movie_ranking.csv, top_user_ranking.csv, full_node_ranking.csv, top_movie_hub_ego.png, movielens_case_study.gml')

## Case Study Summary
This final section explains the MovieLens network story and its main structural insights.

The MovieLens graph is a bipartite network with distinct user and movie node types. The construction pipeline preserves this structure while enriching movies with metadata such as title and genres.

The final metrics show that the network is sparse, connected through a large giant component, and dominated by a small set of high-degree movie hubs. These hubs are the most important spreaders and serve as the primary bridges between the large population of users.

The exported artifacts include a GraphML file for graph visualization tools, ranked CSV tables for the top movies and users, and a presentation-ready ego network figure around the top movie. This package is suitable for a compact case-study submission consistent with Lecture 11.